In [1]:
# install necessary packages:

! pip install requests beautifulsoup4 pandas lxml matplotlib


'pip' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.


In [2]:
import pandas as pd

company_data_df = pd.read_csv(".\companies_ranked_by_revenue.csv")

company_data_df.head(100)

<>:3: SyntaxWarning: invalid escape sequence '\c'
<>:3: SyntaxWarning: invalid escape sequence '\c'
C:\Users\fsbertoglio\AppData\Local\Temp\ipykernel_28772\3276242839.py:3: SyntaxWarning: invalid escape sequence '\c'
  company_data_df = pd.read_csv(".\companies_ranked_by_revenue.csv")


,Unnamed: 0,domain,company_name,sector
0,NaN,cscec.com,China State Construction Engineering,Industrials
1,NaN,samsung.com,Samsung,Information Technology
2,NaN,alibaba.com,Alibaba,Information Technology
3,NaN,mitsubishicorp.com,Mitsubishi Corporation,Industrials
4,NaN,hyundai.com,Hyundai,Consumer Discretionary
...,...,...,...,...
95,NaN,mitsubishi-hc-capital.com,Mitsubishi HC Capital,Financials
96,NaN,henryschein.com,Henry Schein,Health Care
97,NaN,chs.net,Community Health Systems,Health Care
98,NaN,wsp.com,WSP Global,Industrials


FileNotFoundError: [Errno 2] No such file or directory: '.\\extracted_top_companies_brazil_revista_exame.csv'

: 

In [ ]:
import pandas as pd
import numpy as np
import requests
import re
import time
import os
from duckduckgo_search import DDGS
from tqdm import tqdm

SLEEP_BETWEEN_QUERIES = 3  # in seconds
MAX_RETRIES = 3
AUTO_SAVE_EVERY = 5

# --- SEARCH DOMAIN USING DUCKDUCKGO ---
def search_domain(company_name, prompt=""):
    query = f"{company_name} {prompt}"
    try:
        with DDGS() as ddgs:
            results = ddgs.text(query)
            for result in results:
                url = result.get("href")
                if url:
                    domain = re.findall(r"https?://(www\.)?([^/]+)", url)
                    if domain:
                        return domain[0][1]
    except Exception as e:
        print(f"[ERROR] Search failed for '{company_name}': {e}")
    return None


# --- VALIDATE DOMAIN EXISTS (HTTPS) ---
def validate_domain(domain):
    try:
        url = f"https://{domain}"
        response = requests.get(url, timeout=5)
        return response.status_code < 400
    except Exception:
        return False

# --- PROCESS SINGLE COMPANY WITH RETRIES ---
def process_company(company_name, prompt):
    for attempt in range(1, MAX_RETRIES + 1):
        domain = search_domain(company_name, prompt)
        if domain:
            is_valid = validate_domain(domain)
            return {
                "Company": company_name,
                "Domain": domain,
                "Domain_Valid": is_valid
            }
        else:
            print(f"[RETRY] Attempt {attempt} failed for '{company_name}'. Retrying...")
            time.sleep(SLEEP_BETWEEN_QUERIES)
    print(f"[FAILED] Could not resolve domain for '{company_name}' after {MAX_RETRIES} attempts.")
    return {
        "Company": company_name,
        "Domain": "",
        "Domain_Valid": False
    }



In [ ]:
import os
import pandas as pd
import time
from tqdm import tqdm

INPUT_CSV = r".\brazil_companies_financial_data.csv"
OUTPUT_CSV = r".\brazil_domain_data.csv"

PROMPT_TEXT = "site oficial da empresa no brasil"

# --- LOAD COMPANY LIST ---
df_companies = pd.read_csv(INPUT_CSV, sep=";")
company_names = df_companies['company_name'].tolist()
company_sectors = df_companies['sector_primary'].tolist()

# --- IF PARTIAL OUTPUT EXISTS, CONTINUE FROM LAST POINT ---
processed = []
start_index = 0
if os.path.exists(OUTPUT_CSV):
    existing = pd.read_csv(OUTPUT_CSV)
    processed_companies = set(existing['Company'])
    start_index = len(existing)
    processed = existing.to_dict(orient='records')
    print(f"[INFO] Resuming from company #{start_index}...")

# --- MAIN LOOP ---
for idx in tqdm(range(start_index, len(company_names)), desc="Processing companies"):
    company_name = company_names[idx]
    sector = company_sectors[idx]
    prompt = f" {sector} {PROMPT_TEXT}" if sector else PROMPT_TEXT
    result = process_company(company_name, prompt)
    processed.append(result)
    
    # Save after every N companies
    if (idx + 1) % AUTO_SAVE_EVERY == 0 or idx == len(company_names) - 1:
        pd.DataFrame(processed).to_csv(OUTPUT_CSV, index=False)
        print(f"[AUTO-SAVE] Saved progress at {idx + 1} companies.")
    
    time.sleep(SLEEP_BETWEEN_QUERIES)

print(f"\n✅ Process completed. Final results saved to: {OUTPUT_CSV}")


[INFO] Resuming from company #985...


Processing companies:  27%|██▋       | 4/15 [00:18<00:48,  4.37s/it]

[AUTO-SAVE] Saved progress at 990 companies.


Processing companies:  60%|██████    | 9/15 [00:43<00:29,  4.89s/it]

[AUTO-SAVE] Saved progress at 995 companies.


Processing companies:  93%|█████████▎| 14/15 [01:12<00:05,  5.84s/it]

[AUTO-SAVE] Saved progress at 1000 companies.


Processing companies: 100%|██████████| 15/15 [01:17<00:00,  5.17s/it]


✅ Process completed. Final results saved to: .\brazil_domain_data.csv


In [ ]:
br_domain_df = pd.read_csv(r".\brazil_domain_data.csv")
br_domain_df = br_domain_df[["Company", "Domain"]].rename(columns={
    "Company": "company_name",
    "Domain": "domain"
})

br_company_data_df = pd.read_csv(r".\brazil_companies_financial_data.csv", sep=";")

br_full_data_df = br_company_data_df.merge(br_domain_df, on="company_name", how="left").drop_duplicates().reset_index(drop=True)
br_full_data_df['company_name'] = br_full_data_df['company_name'].str.lower()

br_full_data_df.to_csv(r".\brazil_companies_full_data.csv", index=False, sep=";")

br_full_data_df

,company_name,country_name,state_name,city_name,sector_primary,revenue,earnings,total_assets,domain
0,petrobras,Brazil,RJ,RIO DE JANEIRO,Petróleo e Químico,511944000.0,125166000.0,1.050888e+09,petrobras.com.br
1,jbs,Brazil,SP,SAO PAULO,Alimentos e Bebidas,363816537.0,-729145.0,2.061321e+08,jbs.com.br
2,banco do brasil,Brazil,DF,BRASILIA,Bancos,269225339.0,33818951.0,2.172480e+09,bb.com.br
3,raizen combustiveis,Brazil,RJ,RIO DE JANEIRO,Energia,245831790.0,2503326.0,1.115515e+08,raizen.com.br
4,itau unibanco holding,Brazil,SP,SAO PAULO,Bancos,222385000.0,33389000.0,2.543100e+09,itau.com.br
...,...,...,...,...,...,...,...,...,...
995,alumbra,Brazil,SC,Itajaí,Imobiliário e Construção Civil,51461.0,22719.0,9.479500e+04,alumbra.com.br
996,cnp consórcio,Brazil,DF,Brasília,Serviços Financeiros,48294.0,49509.0,6.534910e+05,cnpconsorcio.com.br
997,banco xcmg,Brazil,MG,Pouso Alegre,Bancos,31221.0,7229.0,4.606950e+05,bancoxcmg.com.br
998,eletrobras par,Brazil,RJ,Rio de Janeiro,Energia,26239.0,19130.0,2.290470e+05,eletrobras.com


In [ ]:
INPUT_CSV = r".\eu_companies_financial_data.csv"
OUTPUT_CSV = r".\eu_domain_data.csv"

PROMPT_TEXT = "official website"

# --- LOAD COMPANY LIST ---
df_companies = pd.read_csv(INPUT_CSV, sep=";")
company_names = df_companies['company_name'].tolist()
company_sectors = df_companies['sector_primary'].tolist() if 'sector_primary' in df_companies.columns else [''] * len(company_names)

# --- IF PARTIAL OUTPUT EXISTS, CONTINUE FROM LAST POINT ---
processed = []
start_index = 0
if os.path.exists(OUTPUT_CSV):
    existing = pd.read_csv(OUTPUT_CSV)
    processed_companies = set(existing['Company'])
    start_index = len(existing)
    processed = existing.to_dict(orient='records')
    print(f"[INFO] Resuming from company #{start_index}...")

# --- MAIN LOOP ---
for idx in tqdm(range(start_index, len(company_names)), desc="Processing companies"):
    company_name = company_names[idx]
    sector = company_sectors[idx] if 'company_sectors' in locals() else ''
    prompt = f" {sector} {PROMPT_TEXT}" if sector else PROMPT_TEXT
    result = process_company(company_name, prompt)
    processed.append(result)
    
    # Save after every N companies
    if (idx + 1) % AUTO_SAVE_EVERY == 0 or idx == len(company_names) - 1:
        pd.DataFrame(processed).to_csv(OUTPUT_CSV, index=False)
        print(f"[AUTO-SAVE] Saved progress at {idx + 1} companies.")
    
    time.sleep(SLEEP_BETWEEN_QUERIES)

print(f"\n✅ Process completed. Final results saved to: {OUTPUT_CSV}")


Processing companies:   0%|          | 4/2600 [00:21<3:51:54,  5.36s/it]

[RETRY] Attempt 1 failed for 'TotalEnergies'. Retrying...
[AUTO-SAVE] Saved progress at 5 companies.


Processing companies:   0%|          | 9/2600 [00:55<4:40:51,  6.50s/it]

[AUTO-SAVE] Saved progress at 10 companies.


Processing companies:   1%|          | 14/2600 [01:44<5:59:22,  8.34s/it]

[AUTO-SAVE] Saved progress at 15 companies.


... (1,253 outputs left out)

[AUTO-SAVE] Saved progress at 2600 companies.


Processing companies: 100%|██████████| 2600/2600 [4:49:16<00:00,  6.68s/it]


✅ Process completed. Final results saved to: .\eu_domain_data.csv


: 

In [ ]:
import pandas as pd

eu_domain_df = pd.read_csv(r".\eu_domain_data.csv")
eu_domain_df = eu_domain_df[["Company", "Domain"]].rename(columns={
    "Company": "company_name",
    "Domain": "domain"
})

eu_company_data_df = pd.read_csv(r".\eu_companies_financial_data.csv", sep=";")

eu_full_data_df = eu_company_data_df.merge(eu_domain_df, on="company_name", how="left").drop_duplicates().reset_index(drop=True)
eu_full_data_df['company_name'] = eu_full_data_df['company_name'].str.lower()

eu_full_data_df.to_csv(r".\eu_companies_full_data.csv", index=False, sep=";")

eu_full_data_df

,company_name,country_name,revenue,earnings,total_assets,domain
0,volkswagen,Germany,351872611960,15798600000,6.920490e+11,vw.com
1,shell,United Kingdom,281067000000,27838000000,3.892480e+11,shell.com
2,glencore,Switzerland,230944000000,-998000000,1.304600e+11,glencore.com
3,exor,Netherlands,223206888765,10125000000,4.418450e+10,exor.com
4,totalenergies,France,191626000000,27747000000,2.910570e+11,totalenergies.com
...,...,...,...,...,...,...
2603,greencoat uk wind,United Kingdom,-42208892,-69540800,6.514780e+09,greencoat-ukwind.com
2604,ip group,United Kingdom,-225256315,-259390000,1.444650e+09,ipgroupplc.com
2605,kinnevik,Sweden,-544733164,-545001000,3.930160e+09,kinnevik.com
2606,banque nationale de belgique,Belgium,-4370906478,-3726070000,NaN,nbb.be


In [2]:
# get all unique country_name from all_dns_pipeline_multi_ns_cleaned_with_revenue_sorted

import pandas as pd
all_data_df = pd.read_csv(r".\all_dns_pipeline_multi_ns_cleaned_with_revenue_sorted.csv")
unique_countries = all_data_df['country_name'].dropna().unique().tolist()

unique_countries

#salvar em csv

import pandas as pd
unique_countries = pd.DataFrame(unique_countries, columns=['country_name'])
unique_countries.to_csv(r".\unique_countries.csv", index=False)